In [10]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Model Logic
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, ParameterSampler
# Metrics & Visualization Tools
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report, 
    ConfusionMatrixDisplay
)
from sklearn.manifold import TSNE

# Notebook Configuration
%matplotlib inline
sns.set_style("whitegrid")

In [11]:
# Loading CSVs
csv_files = [
    'embeddings/chinese1.csv', 'embeddings/chinese2.csv',
    'embeddings/korean1.csv', 'embeddings/korean2.csv',
    'embeddings/hindi1.csv', 'embeddings/hindi2.csv',
    'embeddings/spanish1.csv', 'embeddings/spanish2.csv',
    'embeddings/vietnamese1.csv', 'embeddings/vietnamese2.csv',
    'embeddings/arabic1.csv', 'embeddings/arabic2.csv'
]

df_list = [pd.read_csv(f) for f in csv_files]
df_all = pd.concat(df_list, axis=0)

In [12]:
if 'Unnamed: 0' in df_all.columns:
    df_all = df_all.drop(columns=['Unnamed: 0'])

In [13]:
# ones inside csv are 
X_csv = df_all.drop(columns=['Y']).values
y_csv = df_all['Y'].values

In [14]:
X_npy = np.load('XM_L2.npy')
y_npy = np.load('yM_L2.npy')

# Combining both sources
X_combined = np.vstack((X_csv, X_npy))
y_combined = np.concatenate((y_csv, y_npy))

y_combined = np.array([str(label).lower() for label in y_combined])

In [15]:
def run_full_analysis_with_tuning(X, y, title, n_trials=5):
    print(f"\n{'='*20} PROCESSING: {title} {'='*20}")
    
    # 1. Label Encoding
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # 2. The 80/10/10 Split
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )
    print(f"Data split: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")
    
    # 3. Define the Hyperparameter Search Space
    param_grid = {
        'max_depth': [3, 4, 5, 6, 7],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma': [0, 0.1, 0.2, 0.5],
        'min_child_weight': [1, 5, 10, 30]
    }
    
    print(f"\nStarting Auto-Tuning ({n_trials} random configurations)...")
    best_val_score = 0
    best_params = None
    best_model = None
    
    # 4. Auto-Tuning Loop (Train on Train, Evaluate on Val)
    # We use ParameterSampler to pick random combinations from the grid to save time
    for i, params in enumerate(ParameterSampler(param_grid, n_iter=n_trials, random_state=42)):
        
        # Add required fixed parameters
        params['objective'] = 'multi:softprob'
        params['num_class'] = len(le.classes_)
        params['n_estimators'] = 2000
        params['random_state'] = 42
        params['eval_metric'] = 'mlogloss'
        params['early_stopping_rounds'] = 20
        
        # Train candidate model
        candidate_model = XGBClassifier(**params)
        candidate_model.fit(
            X_train, y_train, 
            eval_set=[(X_val, y_val)], 
            verbose=False
        )
        
        # Check performance on Validation set
        val_preds = candidate_model.predict(X_val)
        val_acc = accuracy_score(y_val, val_preds)
        
        print(f"  Trial {i+1}/{n_trials} - Val Acc: {val_acc:.4f} (Depth: {params['max_depth']}, LR: {params['learning_rate']})")
        
        # Save the best model
        if val_acc > best_val_score:
            best_val_score = val_acc
            best_params = params
            best_model = candidate_model
            
    print(f"\n🏆 Best Validation Accuracy: {best_val_score:.4f}")
    print(f"Best Parameters: max_depth={best_params['max_depth']}, learning_rate={best_params['learning_rate']}")
    
    # 5. Generate Predictions on Test Set using the WINNING model
    y_pred = best_model.predict(X_test)
    
    # 6. Print Final Metrics (using Test Set)
    print(f"\n--- FINAL TEST METRICS ({title}) ---")
    print(f"Final Train Accuracy: {best_model.score(X_train, y_train):.4f}")
    print(f"Final Test Accuracy:  {best_model.score(X_test, y_test):.4f}")
    
    weighted_precision = precision_score(y_test, y_pred, average='weighted')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"Final Test Precision (Weighted): {weighted_precision:.4f}")
    print(f"Final Test F1 Score (Weighted):  {weighted_f1:.4f}\n")
    
    print("Classification Report (Test Set):")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    # 7. Plot Confusion Matrix
    plt.figure(figsize=(10, 8))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f"Confusion Matrix (Test Set): {title}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    # 8. Plot t-SNE with XGBoost Decision Boundaries
    print(f"Generating t-SNE plot for {title}...")
    tsne = TSNE(n_components=2, random_state=42)
    X_embedded = tsne.fit_transform(X_test)
    
    h = .02  
    x_min, x_max = X_embedded[:, 0].min() - 1, X_embedded[:, 0].max() + 1
    y_min, y_max = X_embedded[:, 1].min() - 1, X_embedded[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    clf_2d = XGBClassifier(n_estimators=100, max_depth=best_params['max_depth'], learning_rate=best_params['learning_rate'])
    clf_2d.fit(X_embedded, y_test)

    Z = clf_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(12, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    scatter = plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y_test, 
                        edgecolors='k', s=45, cmap='viridis', alpha=0.8)
    plt.legend(handles=scatter.legend_elements()[0], labels=list(le.classes_), 
               title="Classes", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.title(f"t-SNE Visualization (Test Set Boundaries): {title}")
    plt.tight_layout()
    plt.show()

In [16]:
# Analysis 1: Male Voices
run_full_analysis_with_tuning(X_csv, y_csv, "Male Voices Only", n_trials=10)

# Analysis 2: Female Voices
run_full_analysis_with_tuning(X_npy, y_npy, "Female Voices Only", n_trials=10)

# Analysis 3: Combined Dataset
run_full_analysis_with_tuning(X_combined, y_combined, "Combined Dataset (Male + Female)", n_trials=10)


==================== PROCESSING: Male Voices Only ====================
Data split: Train=10736, Val=1342, Test=1342

Starting Auto-Tuning (10 random configurations)...
  Trial 1/10 - Val Acc: 0.9478 (Depth: 4, LR: 0.1)
  Trial 2/10 - Val Acc: 0.9657 (Depth: 5, LR: 0.05)
  Trial 3/10 - Val Acc: 0.9575 (Depth: 7, LR: 0.1)
  Trial 4/10 - Val Acc: 0.9724 (Depth: 4, LR: 0.1)
  Trial 5/10 - Val Acc: 0.9642 (Depth: 4, LR: 0.2)
  Trial 6/10 - Val Acc: 0.9501 (Depth: 3, LR: 0.01)
  Trial 7/10 - Val Acc: 0.9687 (Depth: 6, LR: 0.2)
  Trial 8/10 - Val Acc: 0.9501 (Depth: 6, LR: 0.01)
  Trial 9/10 - Val Acc: 0.9672 (Depth: 5, LR: 0.05)
  Trial 10/10 - Val Acc: 0.9620 (Depth: 6, LR: 0.01)

🏆 Best Validation Accuracy: 0.9724
Best Parameters: max_depth=4, learning_rate=0.1

--- FINAL TEST METRICS (Male Voices Only) ---
Final Train Accuracy: 1.0000
Final Test Accuracy:  0.9709


NameError: name 'precision_score' is not defined

In [8]:
# separate the male and female embeddings
# make separete clusters plots 
# 